# Lab 11: Grid Localization using Bayes Filter (Virtual Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

<hr>


In [6]:
%load_ext autoreload
%autoreload 2

import traceback
from notebook_utils import *
from Traj import *
import asyncio
from localization_extras import Localization

# Setup Logger
LOG = get_logger('demo_notebook.log')

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander

gui.show()

# Start the simulator
START_SIM()

# Start the plotter
START_PLOTTER()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-04-22 13:09:45,178 | INFO     |: Logger demo_notebook.log already initialized.
2026-04-22 13:09:45,178 | INFO     |: GUI is already running. Shutdown notebook to force restart the GUI.


TwoByTwoLayout(children=(Label(value='Simulator', layout=Layout(grid_area='top-left', width='80px')), HBox(chi…

2026-04-22 13:09:45,181 | INFO     |: Creating New Simulator Process


/Users/donghaohong/Documents/FastRobots_ble/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


2026-04-22 13:09:45,793 | INFO     |: Creating New Plotter Process


In [7]:
# Initialize Robot to communicate with the virtual robot and plotter
robot = VirtualRobot(cmdr)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a VirtualRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()

2026-04-22 13:09:49,776 | INFO     |:  | Number of observations per grid cell: 18
2026-04-22 13:09:49,777 | INFO     |:  | Precaching Views...
2026-04-22 13:09:50,681 | INFO     |:  | Precaching Time: 0.903 secs
2026-04-22 13:09:50,681 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 13:09:50,682 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


In [10]:
# Start the simulator
STOP_SIM()

# Start the plotter
STOP_PLOTTER()

2026-04-22 13:11:57,136 | ERROR    |: Simulator is not running
2026-04-22 13:11:57,137 | INFO     |: Plotter is stopped


# Run the Bayes Filter
The cells below utilizes the member functions of class **Localization** (defined in [localization_extras.py](../localization_extras.py)) in each iteration of the Bayes filter algorithm to localize the robot in the grid map. <br>

In [8]:
# Reset Robot and Plots
robot.reset()
cmdr.reset_plotter()

# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.print_update_stats(plot_data=True)

# Plot Odom and GT
current_odom, current_gt = robot.get_pose()
cmdr.plot_gt(current_gt[0], current_gt[1])
cmdr.plot_odom(current_odom[0], current_odom[1])

2026-04-22 13:09:55,166 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-22 13:09:55,166 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-04-22 13:09:58,144 | INFO     |: Update Step
2026-04-22 13:09:58,145 | INFO     |:      | Update Time: 0.001 secs
2026-04-22 13:09:58,146 | INFO     |: ---------- UPDATE STATS -----------
2026-04-22 13:09:58,155 | INFO     |: GT index      : (6, 4, 9)
2026-04-22 13:09:58,155 | INFO     |: Bel index     : (np.int64(5), np.int64(4), np.int64(9)) with prob = 0.8674340
2026-04-22 13:09:58,155 | INFO     |: Bel_bar prob at index = 0.00051440329218107
2026-04-22 13:09:58,156 | INFO     |: GT            : (0.000, 0.000, 360.000)
2026-04-22 13:09:58,156 | INFO     |: Belief        : (0.000, 0.000, 10.000)
2026-04-22 13:09:58,156 | INFO     |: POS ERROR     : (-0.000, -0.000, 350.000)
2026-04-22 13:09:58,156 | INFO     |: ---------- UPDATE STATS -----------


In [9]:
# Initialize the Trajectory object
traj = Trajectory(loc)

# Run through each motion steps
for t in range(0, traj.total_time_steps):
    print("\n\n-----------------", t, "-----------------")
    
    prev_odom, current_odom, prev_gt, current_gt = traj.execute_time_step(t)
        
    # Prediction Step
    loc.prediction_step(current_odom, prev_odom)
    loc.print_prediction_stats(plot_data=True)
    
    # Get Observation Data by executing a 360 degree rotation motion
    loc.get_observation_data()
    
    # Update Step
    loc.update_step()
    loc.print_update_stats(plot_data=True)

# Uncomment the below line to wait for keyboard input between each iteration.
#   input("Press Enter to Continue")
        
    print("-------------------------------------")



----------------- 0 -----------------
2026-04-22 13:10:10,203 | INFO     |: Prediction Step
2026-04-22 13:10:10,222 | INFO     |:  | Prediction Time: 0.018 secs
2026-04-22 13:10:10,222 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-22 13:10:10,237 | INFO     |: GT index         : (6, 3, 7)
2026-04-22 13:10:10,237 | INFO     |: Prior Bel index  : (np.int64(8), np.int64(5), np.int64(8)) with prob = 0.0705766
2026-04-22 13:10:10,237 | INFO     |: POS ERROR        : (-0.633, -0.392, -29.725)
2026-04-22 13:10:10,238 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-22 13:10:13,199 | INFO     |: Update Step
2026-04-22 13:10:13,200 | INFO     |:      | Update Time: 0.001 secs
2026-04-22 13:10:13,200 | INFO     |: ---------- UPDATE STATS -----------
2026-04-22 13:10:13,210 | INFO     |: GT index      : (6, 3, 7)
2026-04-22 13:10:13,211 | INFO     |: Bel index     : (np.int64(6), np.int64(4), np.int64(6)) with prob = 0.9999999
2026-04-22 13:10:13,211 | INFO     |: B

Traceback (most recent call last):
  File "/Users/donghaohong/Documents/ble_robot_1.4/FastRobots-sim-release-main/src/plotter.py", line 263, in keyPressEvent
    if event.key() == Qt.Key_Escape:
                      ^^^^^^^^^^^^^
AttributeError: type object 'Qt' has no attribute 'Key_Escape'
2026-04-22 13:11:41.893 Python[30698:2093219] error messaging the mach port for IMKCFRunLoopWakeUpReliable
Traceback (most recent call last):
  File "/Users/donghaohong/Documents/ble_robot_1.4/FastRobots-sim-release-main/src/plotter.py", line 252, in closeEvent
    result = QtGui.QMessageBox.question(self,
             ^^^^^^^^^^^^^^^^^
AttributeError: module 'pyqtgraph.Qt.QtGui' has no attribute 'QMessageBox'


Loading Flatland...
Initializing pygame framework...
